# SFUMATO — BGM clustering from custom features

This notebook runs the **Bayesian Gaussian Mixture (BGM)** step of SFUMATO
on **your own pre-computed features**, skipping the internal SFUMATO preprocessing.

### What you need
| Variable | Description |
|----------|-------------|
| `X` | Feature matrix — shape `(n_bins, n_features)`, dtype float |
| `coords` | Spatial coordinates — shape `(n_bins, 2)`, columns = `[x, y]` |

Each row is one **spatial bin** (pixel / region). Rows must be aligned: row `i` of `X` corresponds to row `i` of `coords`.

### What you get
For each value of `K` in `K_LIST`:
- **`{RUN_NAME}_K{k}_clusters.csv`** — cluster assignment + colors per bin (simplified output)
- **`{RUN_NAME}_K{k}_probabilities.csv`** — soft-assignment probability for every cluster
- **Inline map** — spatial plot colored by `color_mixed`

### Requirements
- Python environment with SFUMATO dependencies installed
- This notebook must be placed **inside the SFUMATO root folder** (next to `run_bgm_gpu.py`),
  or `SFUMATO_DIR` below must point to it
- A **CUDA-capable GPU** (required by SFUMATO BGM)

---
## 1. Imports & setup

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.decomposition import TruncatedSVD

# ── SFUMATO path ──────────────────────────────────────────────────────────────
# If this notebook lives inside the SFUMATO root folder, leave SFUMATO_DIR = Path(".")
# Otherwise set it to the absolute path of the SFUMATO root folder.
SFUMATO_DIR = Path(".")
sys.path.insert(0, str(SFUMATO_DIR.resolve()))

from run_bgm_gpu import run_bgm
from utils_bgm import BGMConfig, result_dir_for, bgm_stem_for, image_dir_for
from utils_preprocess import make_preprocess_stem

print("Imports OK")

### GPU check
SFUMATO BGM requires a CUDA GPU. The cell below will raise an error if none is found.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise EnvironmentError(
        "No CUDA GPU detected. SFUMATO BGM requires a GPU.\n"
        "Make sure you are on a machine with a CUDA-capable GPU and that "
        "the correct version of PyTorch (with CUDA) is installed."
    )
print(f"GPU: {torch.cuda.get_device_name(0)}")

---
## 2. Configuration

**Edit this cell before running anything else.**

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════

RUN_NAME = "my_experiment"       # label used in all output file names

K_LIST   = [10, 15, 20]          # number(s) of clusters to fit
                                  # BGM is run once and cut at each K — no extra cost

OUTROOT   = Path("./results")    # folder where results will be saved
CACHE_DIR = Path("./sfumato_cache")  # temporary cache folder — created automatically

# ── Dimensionality reduction (SVD) ───────────────────────────────────────────
# If your features are already low-dimensional (e.g. < 50 columns),
# set RUN_SVD = False and they will be used directly.
# If you have many features (e.g. hundreds of genes), set RUN_SVD = True.
RUN_SVD          = False
N_SVD_COMPONENTS = 20     # number of SVD components — only used if RUN_SVD = True

# ── BGM parameters ────────────────────────────────────────────────────────────
BGM_OVERSAMPLE   = 1.5    # fit K_max × BGM_OVERSAMPLE components, then merge
                           # higher = finer initial decomposition, slower
BGM_WEIGHT_PRIOR = 1.0    # Dirichlet process concentration prior
SEED             = 8      # random seed

# ══════════════════════════════════════════════════════════════════════════════

---
## 3. Load your data

Load your feature matrix `X` and spatial coordinates `coords`.

**Two options are shown below — keep only the one that fits your data format.**

| Expected shape | Description |
|----------------|-------------|
| `X` → `(n_bins, n_features)` | Feature matrix, one row per spatial bin |
| `coords` → `(n_bins, 2)` | Spatial coordinates; column 0 = x, column 1 = y |

In [ ]:
# ── Option A: single CSV with x, y columns + feature columns ─────────────────
# df     = pd.read_csv("my_data.csv")
# coords = df[["x", "y"]].values.astype(np.float32)
# X      = df.drop(columns=["x", "y"]).values.astype(np.float32)

# ── Option B: coordinates and features in separate files ─────────────────────
# coords = pd.read_csv("coords.csv").values.astype(np.float32)    # shape (n_bins, 2)
# X      = np.load("features.npy").astype(np.float32)             # shape (n_bins, n_features)
# or:  X = pd.read_csv("features.csv").values.astype(np.float32)

# ─────────────────────────────────────────────────────────────────────────────
# YOUR LOADING CODE BELOW
# ─────────────────────────────────────────────────────────────────────────────





# ─────────────────────────────────────────────────────────────────────────────
# Sanity checks — do not edit
assert X.ndim == 2, \
    "X must be a 2D array (n_bins × n_features)"
assert coords.ndim == 2 and coords.shape[1] == 2, \
    "coords must be a 2D array with exactly 2 columns (x, y)"
assert X.shape[0] == coords.shape[0], \
    f"Row mismatch: X has {X.shape[0]} rows but coords has {coords.shape[0]} rows"

print(f"Loaded {X.shape[0]:,} bins  ×  {X.shape[1]:,} features")
print(f"Coordinate range — x: [{coords[:,0].min():.1f}, {coords[:,0].max():.1f}]  "
      f"y: [{coords[:,1].min():.1f}, {coords[:,1].max():.1f}]")

---
## 4. Dimensionality reduction (optional SVD)

Runs only if `RUN_SVD = True` in the configuration cell.
Skip if your features are already low-dimensional.

In [ ]:
if RUN_SVD:
    print(f"Running TruncatedSVD: {X.shape[1]} → {N_SVD_COMPONENTS} components ...")
    svd   = TruncatedSVD(n_components=N_SVD_COMPONENTS, random_state=SEED)
    X_bgm = svd.fit_transform(X).astype(np.float32)
    expl  = svd.explained_variance_ratio_
    print(f"Variance explained: {expl.sum()*100:.1f}%  ({N_SVD_COMPONENTS} components)")

    fig, ax = plt.subplots(figsize=(6, 3))
    ax.bar(range(1, len(expl) + 1), expl * 100, color="steelblue")
    ax.set_xlabel("Component")
    ax.set_ylabel("Variance explained (%)")
    ax.set_title("SVD — explained variance per component")
    plt.tight_layout()
    plt.show()
else:
    X_bgm = X.astype(np.float32)
    print(f"SVD skipped — using features as-is ({X_bgm.shape[1]} dimensions)")

_COMP = X_bgm.shape[1]   # actual number of dimensions fed into BGM
print(f"BGM input shape: {X_bgm.shape}")

---
## 5. Build SFUMATO cache & run BGM

The cells below prepare the data in the format SFUMATO expects internally,
then call `run_bgm` from SFUMATO unchanged.

**No editing needed from here on.**

In [ ]:
# Build the .npz cache that SFUMATO's run_bgm expects.
# Since there are no individual transcripts in this workflow,
# bins act as both the analysis unit and the "transcript" placeholder.

# Internal dummy values — only affect output directory / file naming
_BIN_WIDTH = 1
_FACTOR    = 1

CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTROOT.mkdir(parents=True, exist_ok=True)

n_bins       = X_bgm.shape[0]
good_bin_ids = np.arange(n_bins, dtype=np.int32)
back_map     = np.arange(n_bins, dtype=np.int32)
gene_bin_id  = np.arange(n_bins, dtype=np.int32)
gene_name    = np.array([f"bin_{i}" for i in range(n_bins)])

stem       = make_preprocess_stem(RUN_NAME, _BIN_WIDTH, _FACTOR, _COMP)
cache_file = CACHE_DIR / f"{stem}.npz"

np.savez_compressed(
    cache_file,
    X_norm       = X_bgm,
    good_bin_ids = good_bin_ids,
    pos          = coords,
    back_map     = back_map,
    gene_x       = coords[:, 0],
    gene_y       = coords[:, 1],
    gene_name    = gene_name,
    gene_bin_id  = gene_bin_id,
    n_bins_total = np.array([n_bins], dtype=np.int32),
)
print(f"Cache saved → {cache_file}")

In [ ]:
config = {
    "run_name":        RUN_NAME,
    "__project_root": str(OUTROOT.resolve()),
    "preprocess": {
        "k_list":           K_LIST,
        "bgm_oversample":   BGM_OVERSAMPLE,
        "bin_width":        _BIN_WIDTH,
        "factor":           _FACTOR,
        "comp":             _COMP,
        "seed":             SEED,
        "save_p2r":         False,   # not applicable in this workflow
        "cache_dir":        str(CACHE_DIR.resolve()),
    },
    "bgm": {
        "outroot":               str(OUTROOT.resolve()),
        "save_transcript_proba": True,   # needed to produce probability maps
        "merge_method":          "complete",
        "merge_metric":          "cosine",
        "bgm_weight_prior":      BGM_WEIGHT_PRIOR,
    },
    "color": {
        "colormap":                    "gist_ncar",
        "colormap_start":              0.08,
        "colormap_end":                0.92,
        "dendrogram_branch_linewidth": 3.0,
    },
}

run_bgm(config)
print("\nBGM complete.")

---
## 6. Post-processing & outputs

For each K, this section produces:

1. **`{RUN_NAME}_K{k}_clusters.csv`** — simplified cluster table (see column legend below)
2. **`{RUN_NAME}_K{k}_probabilities.csv`** — soft-assignment probabilities for all clusters
3. **Inline spatial map** colored by `color_mixed`

> **Note on the cluster CSV:** this is a curated subset of the full SFUMATO bin-level
> output (`_binlevel_FULL.csv`). The full file contains additional columns used in
> spatial transcriptomics workflows (p2r clusters, raw probability scores, colormap
> positions, etc.) that are not relevant here. The columns below are renamed for clarity.

### Column legend — `_clusters.csv`

| Column | Description |
|--------|-------------|
| `x`, `y` | Bin spatial coordinates |
| `cluster` | Index of the dominant cluster (0-based) |
| `cluster_2` | Index of the second-best cluster |
| `uncertainty` | 1 − p₁ — probability mass *not* in the top cluster (0 = fully assigned, close to 1 = ambiguous) |
| `color_hard` | Hex color of the dominant cluster |
| `color_mixed` | Hex color blending top-2 clusters weighted by log-probability — **recommended for visualization** |

### Column legend — `_probabilities.csv`

| Column | Description |
|--------|-------------|
| `x`, `y` | Bin spatial coordinates |
| `prob_cluster_0` … `prob_cluster_{K-1}` | Soft-assignment probability for each cluster (rows sum to 1) |

In [ ]:
cfg    = BGMConfig.from_dict(config)
k_bgm  = int(np.ceil(BGM_OVERSAMPLE * max(K_LIST)))

for k in sorted(int(k) for k in K_LIST):

    outdir = result_dir_for(cfg, k)
    stem   = bgm_stem_for(cfg, k, k_bgm)

    # ── Load raw SFUMATO outputs ──────────────────────────────────────────────
    df_raw   = pd.read_csv(outdir / f"{stem}_binlevel_FULL.csv")
    df_proba = pd.read_csv(outdir / f"{stem}_transcripts_proba_FULL.csv")

    # ── 1. Clean cluster CSV ──────────────────────────────────────────────────
    df_clean = (
        df_raw[["x", "y", "cluster", "second_cluster",
                "compl_p1", "color_hard", "color_mixed"]]   
        .rename(columns={
            "second_cluster": "cluster_2",
            "compl_p1":       "uncertainty", 
        })
    )
    clean_path = outdir / f"{RUN_NAME}_K{k}_clusters.csv"
    df_clean.to_csv(clean_path, index=False)
    print(f"[K={k}] Cluster CSV  → {clean_path}")

    # ── 2. Probability CSV ────────────────────────────────────────────────────
    prob_cols  = [c for c in df_proba.columns if c.startswith("p") and c[1:].isdigit()]
    rename_map = {f"p{i+1}": f"prob_cluster_{i}" for i in range(len(prob_cols))}
    df_prob_clean = (
        df_proba[["x", "y"] + prob_cols]
        .rename(columns=rename_map)
    )
    prob_path = outdir / f"{RUN_NAME}_K{k}_probabilities.csv"
    df_prob_clean.to_csv(prob_path, index=False)
    print(f"[K={k}] Probability CSV → {prob_path}")

    # ── 3. Spatial map ────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.scatter(
        df_clean["x"], df_clean["y"],
        c=df_clean["color_mixed"],
        s=2, linewidths=0,
    )
    ax.set_aspect("equal")
    ax.set_title(f"{RUN_NAME}  —  BGM K={k}  (color_mixed)", fontsize=13)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.invert_yaxis()   # uncomment / comment depending on your coordinate system
    plt.tight_layout()

    img_dir = image_dir_for(cfg, k)
    img_dir.mkdir(parents=True, exist_ok=True)
    map_path = img_dir / f"{RUN_NAME}_K{k}_map.png"
    plt.savefig(map_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"[K={k}] Map saved   → {map_path}\n")

print("Done.")